# Gradients of a tied autoencoder

An autoencoder squeezes each input through a few hidden units and rebuilds it. A tied one
uses the same weight matrix for both halves, which halves the parameters. You derive and
code its gradients in NumPy, check them against central differences, and train it on sensor
readings.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

def numeric_grad(f, A, h=1e-6):
    """Central differences of f() in every entry of the array A. A is nudged in place and restored."""
    G = np.zeros_like(A)
    for idx in np.ndindex(A.shape):
        old = A[idx]
        A[idx] = old + h
        up = f()
        A[idx] = old - h
        down = f()
        A[idx] = old
        G[idx] = (up - down) / (2 * h)
    return G

def relu(Z):
    return np.maximum(0.0, Z)

# 300 readings of 6 sensors on a factory line. Three machines run at varying loads,
# and each sensor picks up a fixed mix of the three, plus a little noise.
rng = np.random.default_rng(3)
loads = rng.uniform(0, 1, size=(300, 3))
mix = rng.uniform(0, 1, size=(3, 6))
X = loads @ mix + 0.02 * rng.normal(size=(300, 6))
print(X.shape, X[:2].round(3))

## Part 1 (coding task): The loss and its gradients

A tied autoencoder compresses each reading $\mathbf{x} \in \mathbb{R}^6$ to $k$ numbers and back with
a single weight matrix $W$ of shape $k \times 6$, used once on the way in and once, transposed, on the
way out. With the readings as the rows of $X$ ($N \times 6$):

$$
Z = X W^\top + \mathbf{b}, \qquad H = \operatorname{ReLU}(Z), \qquad \hat X = H W + \mathbf{c}, \qquad
L = \frac{1}{2N} \sum_{n=1}^{N} \lVert \hat{\mathbf{x}}_n - \mathbf{x}_n \rVert^2 .
$$

Here $\mathbf{b}$ has length $k$ and $\mathbf{c}$ length 6. Write `loss_and_grads(W, b, c, X)`, which
returns `(loss, dW, db, dc)`, each gradient shaped like its parameter. No loop over the rows.

In [ ]:
def loss_and_grads(W, b, c, X):
    """L and its gradients with respect to W, b and c, for the tied autoencoder."""
    # TODO
    raise NotImplementedError

In [ ]:
rng_c = np.random.default_rng(5)
Xs = rng_c.normal(size=(5, 6))
Ws, bs, cs = rng_c.normal(size=(3, 6)), rng_c.normal(size=3), rng_c.normal(size=6)
loss, dW, db, dc = loss_and_grads(Ws, bs, cs, Xs)
for name, G, P in (("dW", dW, Ws), ("db", db, bs), ("dc", dc, cs)):
    assert np.shape(G) == P.shape, f"{name} has shape {np.shape(G)}, expected {P.shape}"

def untied_loss(W_in, W_out):
    Xhat = relu(Xs @ W_in.T + bs) @ W_out + cs
    return 0.5 * np.sum((Xhat - Xs) ** 2) / len(Xs)

assert np.isclose(loss, untied_loss(Ws, Ws)), f"loss {loss:.6f}, expected {untied_loss(Ws, Ws):.6f}"
W_in, W_out = Ws.copy(), Ws.copy()
g_in = numeric_grad(lambda: untied_loss(W_in, W_out), W_in)
g_out = numeric_grad(lambda: untied_loss(W_in, W_out), W_out)
if not np.allclose(dW, g_in + g_out, atol=1e-6):
    if np.allclose(dW, g_out, atol=1e-6):
        raise AssertionError("dW is only the decoder's share: W is also used by the encoder, and that use moves the loss too")
    if np.allclose(dW, g_in, atol=1e-6):
        raise AssertionError("dW is only the encoder's share: W is also used by the decoder, and that use moves the loss too")
    raise AssertionError("dW does not match the numeric gradient")
c_ = cs.copy()
assert np.allclose(dc, numeric_grad(lambda: loss_and_grads(Ws, bs, c_, Xs)[0], c_), atol=1e-6), "dc does not match the numeric gradient"
b_ = bs.copy()
assert np.allclose(db, numeric_grad(lambda: loss_and_grads(Ws, b_, cs, Xs)[0], b_), atol=1e-6), "db does not match the numeric gradient"
print("Part 1 passed")

## Part 2 (coding task): Train it

Write `fit(X, W, b, c, lr, steps)`. Starting from the given parameters (copy them, so the
caller's arrays are left alone), it takes `steps` steps of full-batch gradient descent with
learning rate `lr`, using your `loss_and_grads`, and returns `(W, b, c, losses)`, where
`losses[t]` is the loss before step `t`.

The check trains $k = 3$ units for 2,000 steps from a small random start and compares the
result with the loss of the best constant reconstruction, the column means of $X$.

In [ ]:
def fit(X, W, b, c, lr, steps):
    """Full-batch gradient descent on the tied autoencoder. Returns (W, b, c, losses)."""
    # TODO
    raise NotImplementedError

In [ ]:
rng_f = np.random.default_rng(0)
W0, b0, c0 = 0.3 * rng_f.normal(size=(3, 6)), np.zeros(3), np.zeros(6)
W0_before = W0.copy()
W, b, c, losses = fit(X, W0, b0, c0, lr=0.05, steps=2000)
assert np.array_equal(W0, W0_before), "fit changed the caller's W: copy the parameters first"
assert len(losses) == 2000, f"expected 2000 losses, got {len(losses)}"
baseline = 0.5 * np.sum((X - X.mean(axis=0)) ** 2) / len(X)
final = loss_and_grads(W, b, c, X)[0]
print(f"start {losses[0]:.4f}   end {final:.4f}   column means {baseline:.4f}")
assert losses[-1] < losses[0], "the loss went up: the step should move against the gradient"
assert final < 0.03 * baseline, f"the final loss {final:.4f} is not below 3% of the column-mean loss {baseline:.4f}"
print("Part 2 passed")
ratio = final / baseline
print(f"start {losses[0]:.4f}   end {final:.4f}   column means {baseline:.4f}   ratio {ratio:.2%}")


## Solution

### Part 1: The loss and its gradients

In [ ]:
def loss_and_grads(W, b, c, X):
    """L and its gradients with respect to W, b and c, for the tied autoencoder."""
    N = X.shape[0]
    Z = X @ W.T + b
    H = relu(Z)
    R = H @ W + c - X
    loss = 0.5 * np.sum(R ** 2) / N
    G = R / N                          # dL/dXhat
    dc = G.sum(axis=0)
    dW_out = H.T @ G                   # W as the decoder, Xhat = H W + c
    dZ = (G @ W.T) * (Z > 0)           # back through the decoder, then ReLU'
    db = dZ.sum(axis=0)
    dW_in = dZ.T @ X                   # W as the encoder, Z = X W.T + b
    return loss, dW_out + dW_in, db, dc

**Why it works.** $W$ is used twice, so the loss depends on it along two paths and the chain rule adds
them. The decoder path is an ordinary linear layer, $\hat X = HW + \mathbf{c}$, whose weight gradient
is $H^\top G$ with $G = \partial L/\partial \hat X = (\hat X - X)/N$. The encoder path needs the error
back at $Z$: $G W^\top$, through the decoder, times the ReLU slope. Its weight gradient is
$(\partial L/\partial Z)^\top X$, shaped $k \times 6$ like $W$ because the encoder uses $W^\top$. The
check splits the two by giving the encoder and the decoder separate copies of $W$, and the tied
gradient is the sum of the two copies' gradients. Either share alone is a gradient of a different
model.

### Part 2: Train it

In [ ]:
def fit(X, W, b, c, lr, steps):
    """Full-batch gradient descent on the tied autoencoder. Returns (W, b, c, losses)."""
    W, b, c = W.copy(), b.copy(), c.copy()
    losses = []
    for _ in range(steps):
        loss, dW, db, dc = loss_and_grads(W, b, c, X)
        losses.append(loss)
        W -= lr * dW
        b -= lr * db
        c -= lr * dc
    return W, b, c, losses

**Why it works.** Each step moves every parameter against its gradient, and recording the loss
before the step makes `losses[0]` the loss at the start. The readings are mixes of three machine
loads, so three units can rebuild them almost exactly, and the final loss lands at about 2% of the
column-mean loss. With a gradient that has only one of the two shares of $W$
the same loop would be descending some other function, which is why Part 1 checks the gradient
against the loss rather than trusting that training goes down.